# FleetIQ / SmartTaxi — 03. Dataset Design & Chronological Split

## Objective
Build model-ready partitions (`train.parquet`, `val.parquet`, `test.parquet`) from interim cleaned data.
Enforces **chronological time-based splitting** to strictly eliminate future data leakage.


In [1]:
import sys
from pathlib import Path
sys.path.append('..')

import pandas as pd
from src.utils.io import load_config
from src.data.make_dataset import build_and_save_processed_datasets

config = load_config()
print('Config loaded successfully for project:', config['project']['name'])


Config loaded successfully for project: smart-eta-prediction


## 1. Load Interim Data & Build Chronological Splits
Merges NYC TLC zone centroids and splits temporally into Train (70%), Val (15%), and Test (15%).


In [2]:
train_path, val_path, test_path = build_and_save_processed_datasets(sample_per_month=50000, config=config)
print(f'Train split: {train_path}')
print(f'Validation split: {val_path}')
print(f'Test split: {test_path}')


Train split: C:\Users\user\OneDrive - ESPRIT\Bureau\smart-eta-prediction\smart-eta-prediction\data\processed\train.parquet
Validation split: C:\Users\user\OneDrive - ESPRIT\Bureau\smart-eta-prediction\smart-eta-prediction\data\processed\val.parquet
Test split: C:\Users\user\OneDrive - ESPRIT\Bureau\smart-eta-prediction\smart-eta-prediction\data\processed\test.parquet


## 2. Inspect Split Boundaries & Non-Overlapping Timestamps


In [3]:
train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)
test_df = pd.read_parquet(test_path)

print(f'Train: {len(train_df):,} rows | {train_df["tpep_pickup_datetime"].min()} -> {train_df["tpep_pickup_datetime"].max()}')
print(f'Val:   {len(val_df):,} rows | {val_df["tpep_pickup_datetime"].min()} -> {val_df["tpep_pickup_datetime"].max()}')
print(f'Test:  {len(test_df):,} rows | {test_df["tpep_pickup_datetime"].min()} -> {test_df["tpep_pickup_datetime"].max()}')

assert train_df['tpep_pickup_datetime'].max() < val_df['tpep_pickup_datetime'].min()
assert val_df['tpep_pickup_datetime'].max() < test_df['tpep_pickup_datetime'].min()
print('✅ Zero temporal data leakage verified!')


Train: 138,718 rows | 2025-01-01 00:01:30 -> 2025-07-26 02:46:31
Val:   29,725 rows | 2025-07-26 02:47:28 -> 2025-10-13 17:52:32
Test:  29,726 rows | 2025-10-13 17:53:10 -> 2025-10-31 23:59:09
✅ Zero temporal data leakage verified!
